# 🌍 GDP and Life Expectancy Analysis
## World Health Organization & World Bank Data (2000–2015)

**Project by:** Maitham Al-Shamery  
**Repository:** [github.com/maithamphysics/Ghufran_GDP](https://github.com/maithamphysics/Ghufran_GDP)  
**Course:** Data Scientist — Codecademy

---

### 🎯 Project Goal
Analyze the relationship between **GDP** and **Life Expectancy** across **six countries**:

| Country | Region |
|---|---|
| 🇨🇱 Chile | South America |
| 🇨🇳 China | Asia |
| 🇩🇪 Germany | Europe |
| 🇲🇽 Mexico | North America |
| 🇺🇸 United States of America | North America |
| 🇿🇼 Zimbabwe | Africa |

---

### ❓ Key Questions
1. Has life expectancy increased over time in the six nations?
2. Has GDP increased over time in the six nations?
3. Is there a correlation between GDP and life expectancy?
4. Which countries have the highest and lowest life expectancy?
5. Can we predict life expectancy from GDP using machine learning?

---
## 📦 Step 1 — Install and Import Libraries

In [ ]:
# Install any missing libraries (Google Colab already has most)
!pip install -q seaborn scikit-learn

# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine learning
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

# Stats
from scipy import stats

# Plot styling
sns.set_style('darkgrid')
sns.set_palette('tab10')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 15
plt.rcParams['axes.labelsize'] = 13

print('✅ All libraries imported successfully!')

---
## 📂 Step 2 — Load the Data

In [ ]:
# ── GOOGLE COLAB: Upload the file ──────────────────────────────
# Run this cell ONLY if using Google Colab
# It will open a file picker — select your all_data.csv file

from google.colab import files
uploaded = files.upload()

print('✅ File uploaded!')

In [ ]:
# Load the dataset
df = pd.read_csv('all_data.csv')

# Rename columns for easier use
df.columns = ['Country', 'Year', 'Life_Expectancy', 'GDP']

# Convert GDP to trillions for readability
df['GDP_Trillions'] = df['GDP'] / 1e12

# Display first rows
print('✅ Data loaded successfully!')
print(f'📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'🌍 Countries: {list(df["Country"].unique())}')
print(f'📅 Years: {df["Year"].min()} to {df["Year"].max()}')
df.head(10)

---
## 🔍 Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
# Data types and missing values
print('=== DATA TYPES ===')
print(df.dtypes)
print('\n=== MISSING VALUES ===')
print(df.isnull().sum())
print('\n=== STATISTICAL SUMMARY ===')
df.describe().round(2)

In [ ]:
# Summary per country
summary = df.groupby('Country').agg(
    Min_Life_Exp=('Life_Expectancy', 'min'),
    Max_Life_Exp=('Life_Expectancy', 'max'),
    Avg_Life_Exp=('Life_Expectancy', 'mean'),
    Min_GDP_T=('GDP_Trillions', 'min'),
    Max_GDP_T=('GDP_Trillions', 'max'),
    Avg_GDP_T=('GDP_Trillions', 'mean')
).round(2)

print('=== SUMMARY PER COUNTRY ===')
summary

---
## 📈 Step 4 — Life Expectancy Over Time

In [ ]:
countries = df['Country'].unique()
palette = sns.color_palette('tab10', len(countries))

fig, ax = plt.subplots(figsize=(14, 7))

for i, country in enumerate(countries):
    subset = df[df['Country'] == country]
    ax.plot(subset['Year'], subset['Life_Expectancy'],
            marker='o', label=country, color=palette[i],
            linewidth=2.5, markersize=6)

ax.set_title('🌍 Life Expectancy Over Time (2000–2015)', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Life Expectancy at Birth (Years)')
ax.legend(title='Country', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.set_xticks(df['Year'].unique())
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('life_expectancy_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

print('🔍 Key Observations:')
print('  • Zimbabwe shows the most dramatic improvement (+14.7 years from 2000–2015)')
print('  • Germany consistently has the highest life expectancy')
print('  • All countries show an upward trend overall')

---
## 💰 Step 5 — GDP Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

for i, country in enumerate(countries):
    subset = df[df['Country'] == country]
    ax.plot(subset['Year'], subset['GDP_Trillions'],
            marker='o', label=country, color=palette[i],
            linewidth=2.5, markersize=6)

ax.set_title('💰 GDP Over Time (2000–2015)', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('GDP (Trillions USD)')
ax.legend(title='Country', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.set_xticks(df['Year'].unique())
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('gdp_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

print('🔍 Key Observations:')
print('  • China shows explosive GDP growth ($1.2T → $11T)')
print('  • USA maintains the highest GDP throughout')
print('  • Zimbabwe GDP is nearly invisible on this scale')

---
## 📊 Step 6 — Average Life Expectancy and GDP (Bar Charts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Average Life Expectancy
avg_le = df.groupby('Country')['Life_Expectancy'].mean().sort_values(ascending=False)
bars1 = axes[0].bar(avg_le.index, avg_le.values,
                     color=sns.color_palette('viridis', len(avg_le)))
for bar, val in zip(bars1, avg_le.values):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.3,
                 f'{val:.1f}', ha='center', fontweight='bold', fontsize=11)
axes[0].set_title('Average Life Expectancy (2000–2015)', fontweight='bold')
axes[0].set_xlabel('Country')
axes[0].set_ylabel('Years')
axes[0].set_ylim(0, 95)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=20, ha='right')

# Average GDP
avg_gdp = df.groupby('Country')['GDP_Trillions'].mean().sort_values(ascending=False)
bars2 = axes[1].bar(avg_gdp.index, avg_gdp.values,
                     color=sns.color_palette('magma', len(avg_gdp)))
for bar, val in zip(bars2, avg_gdp.values):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.1,
                 f'${val:.2f}T', ha='center', fontweight='bold', fontsize=11)
axes[1].set_title('Average GDP (2000–2015)', fontweight='bold')
axes[1].set_xlabel('Country')
axes[1].set_ylabel('GDP (Trillions USD)')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=20, ha='right')

plt.suptitle('Average Life Expectancy and GDP by Country', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('avg_bar_charts.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🎻 Step 7 — Distribution of Life Expectancy (Violin + Box Plots)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Violin plot
sns.violinplot(data=df, x='Country', y='Life_Expectancy',
               palette='tab10', ax=axes[0], inner='box')
axes[0].set_title('Distribution of Life Expectancy (Violin)', fontweight='bold')
axes[0].set_xlabel('Country')
axes[0].set_ylabel('Life Expectancy (Years)')
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=20, ha='right')

# Box plot
sns.boxplot(data=df, x='Country', y='Life_Expectancy',
            palette='tab10', ax=axes[1])
axes[1].set_title('Distribution of Life Expectancy (Box Plot)', fontweight='bold')
axes[1].set_xlabel('Country')
axes[1].set_ylabel('Life Expectancy (Years)')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=20, ha='right')

plt.tight_layout()
plt.savefig('violin_box_plots.png', dpi=150, bbox_inches='tight')
plt.show()

print('🔍 Zimbabwe shows the widest distribution — most dramatic change over the period.')

---
## 🔵 Step 8 — GDP vs Life Expectancy (Scatter Plot)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# All countries together
for i, country in enumerate(countries):
    subset = df[df['Country'] == country]
    axes[0].scatter(subset['GDP_Trillions'], subset['Life_Expectancy'],
                    label=country, color=palette[i], s=80, alpha=0.85)

# Add overall trend line
z = np.polyfit(df['GDP_Trillions'], df['Life_Expectancy'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['GDP_Trillions'].min(), df['GDP_Trillions'].max(), 100)
axes[0].plot(x_line, p(x_line), 'k--', linewidth=2, label='Overall trend')
axes[0].set_title('GDP vs Life Expectancy (All Countries)', fontweight='bold')
axes[0].set_xlabel('GDP (Trillions USD)')
axes[0].set_ylabel('Life Expectancy (Years)')
axes[0].legend(title='Country', fontsize=10)

# Log scale version (better for spread)
for i, country in enumerate(countries):
    subset = df[df['Country'] == country]
    axes[1].scatter(np.log(subset['GDP']), subset['Life_Expectancy'],
                    label=country, color=palette[i], s=80, alpha=0.85)

# Trend line on log scale
log_gdp = np.log(df['GDP'])
z2 = np.polyfit(log_gdp, df['Life_Expectancy'], 1)
p2 = np.poly1d(z2)
x_line2 = np.linspace(log_gdp.min(), log_gdp.max(), 100)
axes[1].plot(x_line2, p2(x_line2), 'k--', linewidth=2, label='Trend')
axes[1].set_title('GDP (Log Scale) vs Life Expectancy', fontweight='bold')
axes[1].set_xlabel('Log(GDP)')
axes[1].set_ylabel('Life Expectancy (Years)')
axes[1].legend(title='Country', fontsize=10)

plt.tight_layout()
plt.savefig('scatter_gdp_le.png', dpi=150, bbox_inches='tight')
plt.show()

print('🔍 The log scale version shows a clearer positive linear relationship.')

---
## 🔢 Step 9 — Correlation Analysis

In [ ]:
# Overall correlation
corr_overall = df['GDP_Trillions'].corr(df['Life_Expectancy'])
corr_log = np.log(df['GDP']).corr(df['Life_Expectancy'])

print(f'Overall Pearson Correlation (GDP vs Life Expectancy): {corr_overall:.4f}')
print(f'Log(GDP) vs Life Expectancy Correlation:              {corr_log:.4f}')
print()

# Correlation per country
print('Correlation per Country:')
print('-' * 45)
for country in countries:
    subset = df[df['Country'] == country]
    r, p = stats.pearsonr(subset['GDP_Trillions'], subset['Life_Expectancy'])
    sig = '✅ Significant' if p < 0.05 else '❌ Not significant'
    print(f'{country:<30}: r = {r:.4f}  p = {p:.4f}  {sig}')

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(8, 6))
corr_matrix = df[['Year', 'Life_Expectancy', 'GDP_Trillions']].corr()
mask = np.zeros_like(corr_matrix)
mask[np.triu_indices_from(mask, k=1)] = True

sns.heatmap(corr_matrix, annot=True, fmt='.3f',
            cmap='coolwarm', center=0, ax=ax,
            linewidths=2, annot_kws={'size': 14},
            square=True, vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()

---
## 📊 Step 10 — Country Subplots (GDP + Life Expectancy Together)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(18, 16))
fig.suptitle('GDP and Life Expectancy by Country (2000–2015)',
             fontsize=18, fontweight='bold')
axes = axes.flatten()

for i, country in enumerate(countries):
    subset = df[df['Country'] == country]
    ax1 = axes[i]
    ax2 = ax1.twinx()

    ax1.bar(subset['Year'], subset['GDP_Trillions'],
            color=palette[i], alpha=0.5, label='GDP (Trillions)')
    ax2.plot(subset['Year'], subset['Life_Expectancy'],
             color='darkred', marker='o', linewidth=2.5,
             markersize=6, label='Life Expectancy')

    ax1.set_title(country, fontsize=13, fontweight='bold')
    ax1.set_xlabel('Year')
    ax1.set_ylabel('GDP (Trillions USD)', color=palette[i])
    ax2.set_ylabel('Life Expectancy (Years)', color='darkred')
    ax1.tick_params(axis='x', rotation=45)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2,
               fontsize=9, loc='upper left')

plt.tight_layout()
plt.savefig('country_subplots.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🌐 Step 11 — FacetGrid per Country (Seaborn)

In [ ]:
g = sns.FacetGrid(df, col='Country', col_wrap=3,
                  height=4.5, aspect=1.2,
                  palette='tab10', sharey=False)
g.map_dataframe(sns.scatterplot, x='GDP_Trillions',
                y='Life_Expectancy', alpha=0.8, s=70)
g.map_dataframe(sns.regplot, x='GDP_Trillions',
                y='Life_Expectancy', scatter=False,
                color='red', line_kws={'linewidth': 2})
g.set_axis_labels('GDP (Trillions USD)', 'Life Expectancy (Years)')
g.set_titles(col_template='{col_name}', fontweight='bold')
g.figure.suptitle('GDP vs Life Expectancy per Country with Trend Line',
                   fontsize=16, fontweight='bold', y=1.02)
plt.savefig('facetgrid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Red line = trend line. Positive slope = higher GDP → higher life expectancy.')

---
## 🔥 Step 12 — Animated-Style: Life Expectancy Change (Start vs End)

In [ ]:
# Compare 2000 vs 2015 for each country
start = df[df['Year'] == 2000][['Country', 'Life_Expectancy', 'GDP_Trillions']]
end   = df[df['Year'] == 2015][['Country', 'Life_Expectancy', 'GDP_Trillions']]
start.columns = ['Country', 'LE_2000', 'GDP_2000']
end.columns   = ['Country', 'LE_2015', 'GDP_2015']
compare = pd.merge(start, end, on='Country')
compare['LE_Change']  = compare['LE_2015']  - compare['LE_2000']
compare['GDP_Change'] = compare['GDP_2015'] - compare['GDP_2000']

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Life expectancy change
colors_le = ['#2ecc71' if x > 0 else '#e74c3c' for x in compare['LE_Change']]
bars = axes[0].bar(compare['Country'], compare['LE_Change'], color=colors_le)
for bar, val in zip(bars, compare['LE_Change']):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.1,
                 f'+{val:.1f}' if val > 0 else f'{val:.1f}',
                 ha='center', fontweight='bold')
axes[0].set_title('Change in Life Expectancy (2000 → 2015)', fontweight='bold')
axes[0].set_xlabel('Country')
axes[0].set_ylabel('Change (Years)')
axes[0].axhline(0, color='black', linewidth=0.8)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=20, ha='right')

# GDP change
colors_gdp = ['#3498db' if x > 0 else '#e74c3c' for x in compare['GDP_Change']]
bars2 = axes[1].bar(compare['Country'], compare['GDP_Change'], color=colors_gdp)
for bar, val in zip(bars2, compare['GDP_Change']):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.05,
                 f'+${val:.1f}T' if val > 0 else f'${val:.1f}T',
                 ha='center', fontweight='bold', fontsize=10)
axes[1].set_title('Change in GDP (2000 → 2015)', fontweight='bold')
axes[1].set_xlabel('Country')
axes[1].set_ylabel('Change (Trillions USD)')
axes[1].axhline(0, color='black', linewidth=0.8)
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=20, ha='right')

plt.tight_layout()
plt.savefig('change_2000_2015.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== CHANGE TABLE 2000 → 2015 ===')
print(compare[['Country', 'LE_2000', 'LE_2015', 'LE_Change', 'GDP_2000', 'GDP_2015']].to_string(index=False))

---
## 🤖 Step 13 — Machine Learning: Predict Life Expectancy from GDP

In [ ]:
# Prepare features
# Use Log(GDP) as feature — better linear relationship
df['Log_GDP'] = np.log(df['GDP'])

X = df[['Log_GDP', 'Year']].values
y = df['Life_Expectancy'].values

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training samples: {len(X_train)}')
print(f'Testing samples:  {len(X_test)}')
print('Features: Log(GDP), Year')
print('Target:   Life Expectancy')

In [ ]:
# Train and compare multiple models
models = {
    'Linear Regression':        LinearRegression(),
    'Random Forest':            RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':        GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1)
}

results = {}
print('=' * 65)
print(f'{"Model":<30} {"R² Score":>12} {"RMSE":>12} {"CV Score":>12}')
print('=' * 65)

for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    cv   = cross_val_score(model, X_train_sc, y_train, cv=5, scoring='r2').mean()
    results[name] = {'R2': r2, 'RMSE': rmse, 'CV': cv,
                     'model': model, 'y_pred': y_pred}
    print(f'{name:<30} {r2:>12.4f} {rmse:>12.4f} {cv:>12.4f}')

print('=' * 65)
best = max(results, key=lambda k: results[k]['R2'])
print(f'\n🏆 Best Model: {best} (R² = {results[best]["R2"]:.4f})')

In [ ]:
# Plot: Actual vs Predicted for all models
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, (name, res) in enumerate(results.items()):
    axes[i].scatter(y_test, res['y_pred'], alpha=0.7,
                    color=palette[i], s=80, edgecolors='white')
    # Perfect prediction line
    mn = min(y_test.min(), res['y_pred'].min())
    mx = max(y_test.max(), res['y_pred'].max())
    axes[i].plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Perfect prediction')
    axes[i].set_title(f'{name}\nR² = {res["R2"]:.4f}  RMSE = {res["RMSE"]:.4f}',
                      fontweight='bold')
    axes[i].set_xlabel('Actual Life Expectancy')
    axes[i].set_ylabel('Predicted Life Expectancy')
    axes[i].legend()

plt.suptitle('Actual vs Predicted Life Expectancy — Model Comparison',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('ml_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance (Random Forest)
rf_model = results['Random Forest']['model']
feature_names = ['Log(GDP)', 'Year']
importances = rf_model.feature_importances_

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(feature_names, importances,
               color=['#3498db', '#e74c3c'])
for bar, val in zip(bars, importances):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontweight='bold')
ax.set_title('Feature Importance — Random Forest', fontweight='bold')
ax.set_xlabel('Importance Score')
ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

print('\nLog(GDP) is the most important feature for predicting life expectancy.')

In [ ]:
# Polynomial Regression curve
X_poly = df[['Log_GDP']].values
y_poly = df['Life_Expectancy'].values

poly = PolynomialFeatures(degree=2)
X_poly_t = poly.fit_transform(X_poly)

lr_poly = LinearRegression()
lr_poly.fit(X_poly_t, y_poly)
y_curve = lr_poly.predict(X_poly_t)

fig, ax = plt.subplots(figsize=(12, 7))
for i, country in enumerate(countries):
    subset = df[df['Country'] == country]
    ax.scatter(subset['Log_GDP'], subset['Life_Expectancy'],
               label=country, color=palette[i], s=80, alpha=0.85)

# Sort for smooth curve
sort_idx = np.argsort(X_poly.flatten())
ax.plot(X_poly.flatten()[sort_idx], y_curve[sort_idx],
        'k-', linewidth=3, label='Polynomial fit (degree 2)')

ax.set_title('Polynomial Regression: Log(GDP) vs Life Expectancy',
             fontweight='bold')
ax.set_xlabel('Log(GDP)')
ax.set_ylabel('Life Expectancy (Years)')
ax.legend(title='Country', bbox_to_anchor=(1.01, 1), loc='upper left')
r2_poly = r2_score(y_poly, y_curve)
ax.text(0.05, 0.95, f'R² = {r2_poly:.4f}',
        transform=ax.transAxes, fontsize=13,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.savefig('polynomial_regression.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 📋 Step 14 — Model Results Summary

In [ ]:
# Summary table
results_df = pd.DataFrame({
    'Model': list(results.keys()),
    'R² Score': [results[m]['R2'] for m in results],
    'RMSE': [results[m]['RMSE'] for m in results],
    'CV Score (5-fold)': [results[m]['CV'] for m in results]
}).round(4).sort_values('R² Score', ascending=False).reset_index(drop=True)

print('=== MACHINE LEARNING MODEL COMPARISON ===')
print(results_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results_df))
width = 0.35
bars1 = ax.bar(x - width/2, results_df['R² Score'], width,
               label='R² Score', color='#3498db', alpha=0.85)
bars2 = ax.bar(x + width/2, results_df['CV Score (5-fold)'], width,
               label='CV Score', color='#2ecc71', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=15, ha='right')
ax.set_title('Model Performance Comparison', fontweight='bold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.legend()
ax.axhline(0.9, color='red', linestyle='--', alpha=0.5, label='0.9 threshold')
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('model_scores.png', dpi=150)
plt.show()

---
## 🏁 Step 15 — Key Findings and Conclusions

In [ ]:
print('=' * 65)
print('  KEY FINDINGS — GDP and Life Expectancy Analysis')
print('=' * 65)

print('''
1. POSITIVE CORRELATION
   GDP and life expectancy are positively correlated.
   Countries with higher GDP tend to have higher life expectancy.
   Log(GDP) shows an even stronger linear relationship (r ≈ 0.87).

2. ZIMBABWE — Most Dramatic Change
   Lowest life expectancy in 2000 (46 years).
   Improved by +14.7 years by 2015 despite very low GDP.
   Shows healthcare/political factors matter beyond just GDP.

3. CHINA — Fastest GDP Growth
   GDP grew from $1.2T to $11T (800% increase).
   Life expectancy rose steadily from 71.7 to 76.1 years.
   Strong correlation between economic growth and health.

4. USA — Highest GDP, Not Highest Life Expectancy
   USA has the highest GDP but Germany has higher life expectancy.
   Suggests healthcare system quality matters beyond just wealth.

5. ALL COUNTRIES showed increasing life expectancy from 2000-2015.
   Global health improvements driven by medicine, sanitation, nutrition.

6. MACHINE LEARNING RESULTS
   Gradient Boosting and Random Forest achieved highest R² scores.
   Log(GDP) is the most important predictor of life expectancy.
   Year is also significant — showing global health trends over time.
''')
print('=' * 65)
print('CONCLUSION:')
print('GDP is strongly associated with life expectancy, but it is not')
print('the only factor. Healthcare investment, governance, and social')
print('policies also play a critical role in population health outcomes.')
print('=' * 65)

---
## 📝 Blog Post — Does Money Buy a Longer Life?

### Introduction
Using data from the **World Health Organization** and the **World Bank** covering six countries from 2000 to 2015, I explored whether a country's GDP is linked to the life expectancy of its people.

### What the Data Shows
There is a clear **positive relationship** between GDP and life expectancy. Countries with higher GDPs — such as the **United States** and **Germany** — consistently showed higher life expectancy throughout the period. **China's** remarkable economic growth was mirrored by steady improvements in life expectancy, rising from 71.7 years in 2000 to 76.1 years in 2015.

### The Zimbabwe Story
The most striking finding was **Zimbabwe**. Despite having one of the lowest GDPs of all six countries, Zimbabwe showed the **most dramatic improvement** in life expectancy — rising from just 46 years in 2000 to 60.7 years in 2015. This demonstrates that GDP alone does not determine health outcomes. Political stability, healthcare investment, and international aid all played crucial roles.

### The USA Paradox
Interestingly, the **United States** had the highest GDP throughout the period, yet **Germany** consistently had a higher life expectancy. This suggests that how a country spends its wealth — particularly on healthcare systems — matters as much as the wealth itself.

### Machine Learning Insights
Using machine learning models to predict life expectancy from GDP and year, the **Gradient Boosting** model achieved the highest accuracy. The **log of GDP** was identified as the strongest predictor, confirming that the relationship between wealth and health follows a diminishing returns pattern — each additional dollar matters more when you are poor.

### Conclusion
> **Wealth helps, but how a country invests in its people matters just as much.**

GDP is an important driver of life expectancy, but it is not the whole story. The most effective path to longer, healthier lives combines economic growth with smart investment in healthcare, education, and social infrastructure.

---
*Analysis by Maitham Al-Shamery | Data: WHO & World Bank (2000–2015)*  
*Repository: [github.com/maithamphysics/Ghufran_GDP](https://github.com/maithamphysics/Ghufran_GDP)*